In [ ]:
!date

### goal of notebook: summarize proportion of cells lost per QC step, per sample

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))

In [ ]:
!ls -lt {projdir}/h5ad/rna/raw/ambimux/wgs

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/ambimux/wgs/merged.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['sample'] = [x.split('_')[1] for x in adata.obs.index]

In [ ]:
np.sum(adata.obs['pct_counts_mt'] > 30)

In [ ]:
!ls -lt '{projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv'

### load ambimux

In [ ]:
!ls -lt {projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv

In [ ]:
ambimux = pd.read_csv(f'{projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv', sep='\t', header=0, index_col=0)
ambimux.shape

In [ ]:
ambimux['assignment'].value_counts()

In [ ]:
adata.shape, ambimux.shape

In [ ]:
sum(adata.obs.index == ambimux.index), \
sum(adata.obs.index.isin(ambimux.index))

In [ ]:
adata.obs['assignment'] = ambimux.reindex(adata.obs.index)['assignment'].copy()
adata.obs['ambient_rna'] = ambimux.reindex(adata.obs.index)['best_rna_ambient'].copy()
adata.obs['ambient_dna'] = ambimux.reindex(adata.obs.index)['best_atac_ambient'].copy()

### get donor to physical time mapping

In [ ]:
demux_time = pd.read_csv(f'{projdir}/txt/demux_time_map.txt', sep='\t', header=0, index_col=None)
demux_map = dict(zip(demux_time['Sample'], demux_time['TimeSample2']))

adata.obs['sample_donor'] = [f'{x}_{y}' for x,y in zip(adata.obs['sample'], adata.obs['assignment'])]
adata.obs['time'] = [demux_map[x].split('_')[1] for x in adata.obs['sample_donor']]
adata.obs['time_num'] = [int(x[1:]) for x in adata.obs['time']]
adata.obs['donor_time'] = [f'{x}_{y}' for x,y in zip(adata.obs['assignment'], adata.obs['time'])]

In [ ]:
adata.obs['time'].head(), \
adata.obs['time_num'].head()

### load dblfinder

In [ ]:
!ls -lt {projdir}/csv/doublets/rna

In [ ]:
dblfinder = pd.read_csv(f'{projdir}/csv/doublets/rna/ambimux_doublet_scores.csv', sep='\t', header=0, index_col=0)
dblfinder.shape

In [ ]:
dblfinder.head()

In [ ]:
dbl_dist = dblfinder['scDblFinder_DropletType'].value_counts()
dbl_dist / dblfinder.shape[0]

In [ ]:
### check if same order
sum(dblfinder.index == adata.obs.index)

In [ ]:
### aim for 2.5% doublets because we already removed them with ambimux
percentile = 0.975
dbl_cutoff = dblfinder['scDblFinder_Score'].quantile(percentile)
mask = dblfinder['scDblFinder_Score'] > dbl_cutoff
dblfinder['is_doublet'] = mask
sum(mask) / dblfinder.shape[0]

In [ ]:
adata.obs['doublet_score'] = dblfinder.reindex(adata.obs.index)['scDblFinder_Score']
adata.obs['is_doublet'] = dblfinder.reindex(adata.obs.index)['is_doublet']
adata.obs['is_doublet'].replace({'singlet' : False, 'doublet' : True}, inplace=True)
adata.obs['is_doublet'].value_counts()

### plot metadata

In [ ]:
adata.obs.columns

In [ ]:
select_cols = ['n_genes_by_counts', 'total_counts',
               'pct_counts_mt', 'doublet_score', 'ambient_rna']

In [ ]:
xlims = [5000, 1.0, 100, 1.0, 1.0]
fig, axes = plt.subplots(2, 3, figsize=(6, 5))

for i, ax in enumerate(axes.flatten()):
    if i < 5:        
        col = select_cols[i]
        sns.histplot(data=adata.obs, x=col, ax=ax)
        ax.grid(False)
        xlim = xlims[i]
        ax.set_xlim((0, xlim))
        
plt.subplots_adjust(wspace=0.4, hspace=0.4)

In [ ]:
tmp_cols = np.tile(select_cols, 10)
tmp_samples = np.repeat(samples, 5)

xlims = np.tile([5000, 10000, 50, 1.0, 1.0], 10)

fig, axes = plt.subplots(10, 5, figsize=(14, 20))

for i, ax in enumerate(axes.flatten()):
    col = tmp_cols[i]
    s = tmp_samples[i]
    tmp_meta = adata[adata.obs['sample'] == s].obs
    sns.histplot(data=tmp_meta, x=col, ax=ax)
    xlim = xlims[i]
    ax.set_xlim(0, xlim)
    
    ax.grid(False)
#     ax.set_xlabel(ax.get_xlabel(), fontsize=10)
    ax.set_ylabel('')
#     ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
#     ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
    ax.set_title(s)
    
plt.subplots_adjust(wspace=0.5, hspace=1.0)

In [ ]:
### some default options
n_genes_by_counts_filter = 200
total_counts_filter = 500
pct_counts_mt_filter = 30
ambient_rna_filter = 0.50

check_qc = pd.DataFrame(columns=select_cols, index=samples)
for s in samples:
    tmp_meta = adata[adata.obs['sample'] == s].obs
    add = [np.sum(tmp_meta['n_genes_by_counts'] < n_genes_by_counts_filter),
           np.sum(tmp_meta['total_counts'] < total_counts_filter),
        np.sum(tmp_meta['pct_counts_mt'] > pct_counts_mt_filter),
        np.sum(tmp_meta['is_doublet']), 
        np.sum(tmp_meta['ambient_rna'] > ambient_rna_filter)]
    check_qc.loc[s] = add
check_qc

In [ ]:
check_qc

In [ ]:
n_drops = adata.obs['sample'].value_counts().sort_index()
prop_qc = pd.DataFrame([check_qc.iloc[i,:] / n_drops[i] for i in range(len(samples))])
prop_qc

In [ ]:
n_genes_by_counts_filter = 200
total_counts_filter = 500
pct_counts_mt_filter = 30
ambient_rna_filter = 0.5

meta = adata.obs.copy()
all_mask = (meta['n_genes_by_counts'] > n_genes_by_counts_filter) & \
           (meta['total_counts'] > total_counts_filter) & \
        (meta['pct_counts_mt'] < pct_counts_mt_filter) & \
        (~meta['is_doublet']) & \
        (meta['ambient_rna'] < ambient_rna_filter)

n_filtered = np.sum(all_mask)
n_filtered

In [ ]:
n_filtered / adata.obs.shape[0]

In [ ]:
np.sum(meta['n_genes_by_counts'] > n_genes_by_counts_filter) , \
np.sum(meta['total_counts'] > total_counts_filter) , \
np.sum(meta['pct_counts_mt'] < pct_counts_mt_filter) , \
np.sum(~meta['is_doublet']) , \
np.sum(meta['ambient_rna'] < ambient_rna_filter)

In [ ]:
filtered_adata = adata[all_mask]
filtered_adata.shape

In [ ]:
filtered_adata.write_h5ad(f'{projdir}/h5ad/rna/raw/filtered.h5ad')

### write csv files out

In [ ]:
# adata.obs.to_csv(f'{projdir}/csv/qc/rna/wgs/rna_qc_all.csv', sep='\t', header=True, index=True)
filtered_adata.obs.to_csv(f'{projdir}/csv/qc/rna/rna_qc_filtered.csv', sep='\t', header=True, index=True)

In [ ]:
!date